# BP2 Gate 4 — Statistical Validation & Explainability
**Customer360 Navigator Enterprise Suite — Customer Friction Classification**

## Purpose
Implements the same governance step BP1 Gate 4 did for this project's second business problem:
independent-style statistical validation of Gate 3's champion (is it *actually* better than the
runner-up, not just numerically higher on one run?) plus explainability (SHAP) and a supplementary
threshold-independent metric (4-class one-vs-rest macro ROC-AUC). Gate 3's only job was champion
selection by mean CV F1-macro — this gate is where that selection gets scrutinized, not assumed.

## Real Gate 3 result this gate builds on
Gate 3's real run (2026-09-22, on the hardened notebook) completed with no crash, all 10 integrity
checks passed. Champion = `xgboost` (held-out test f1_macro=0.4559, class imbalance ratio
152.0:1). One candidate, `catboost`, failed — the user explicitly declined to investigate the
failure and asked to proceed here. This is not a blocker: champion/runner-up eligibility is read
LIVE from `gate3_cv_benchmark_results.csv` and only ever considers candidates with `status=="OK"`,
the same design BP1 Gate 3 already used for its own candidate failures. Champion and runner-up are
never hardcoded in this notebook — it re-reads Gate 3's real results at run time, so it works
correctly on any future re-run even if the champion changes.

## What "statistical validation" means here, concretely
Gate 3 recorded only the *mean and std* of CV F1-macro per candidate, not individual fold scores —
no paired significance test was possible from Gate 3's artifacts alone. This gate re-runs the
identical `StratifiedKFold` split for the champion and runner-up (read live, never guessed),
captures the paired per-fold F1-macro scores for each, and:
1. Cross-checks its own recomputed mean CV F1-macro for the champion against Gate 3's recorded
   value (within floating-point tolerance) — guards against this notebook's feature-engineering
   code silently drifting from Gate 3's over time.
2. Runs a paired t-test AND a Wilcoxon signed-rank test (nonparametric cross-check) on the paired
   fold scores.
3. Bootstraps a 95% confidence interval for the champion's held-out test F1-macro (1,000 resamples,
   seeded for reproducibility).
4. Computes 4-class one-vs-rest macro ROC-AUC on the held-out test set.

**Honest limitation, stated plainly**: 5 CV folds is a very small sample for a t-test/Wilcoxon
test. This gate reports the numbers as *directional* evidence, not a definitive, high-powered
statistical claim.

## Why this re-run is sequential, not concurrent (relevant after BP2 Gate 3's real crash + Lesson #21)
Unlike Gate 3's 6-candidate benchmark loop, this gate fits only 2 models (champion + runner-up),
one fold at a time, with **no joblib parallel backend at all** — the concurrency-driven memory risk
Lesson #21 hardened Gate 3 against does not apply to this loop by construction. The same monitoring
discipline is still applied given this is the same real, much-larger-than-BP1 dataset: live RAM
headroom logged before every fold, `assert_within_ram_ceiling()` re-checked after every fold,
densified per-fold matrices explicitly freed, and a 15-second precautionary pause between the
champion's and runner-up's re-fits (each is a full real-scale 5-fold fit, not a fast synthetic one).

## SHAP explainability
`shap.LinearExplainer` for a linear champion (accepts sparse input directly) or
`shap.TreeExplainer` for a tree-ensemble champion (RandomForest / HistGradientBoosting / XGBoost /
LightGBM / CatBoost — covers every Gate 3 candidate), selected automatically by the champion's
actual model type. **Carries over a real environment finding from BP1 Gate 4**: this installed
`shap`'s `TreeExplainer.shap_values()` rejects sparse input for every tree-based model, not only
the ones Gate 3's own `NEEDS_DENSE` set names (that set is a separate, fit-time-only constraint) —
so the sample/background matrix is always densified before `TreeExplainer`, regardless of
`NEEDS_DENSE` membership. Missing this distinction here would have caused a real SHAP failure on
this exact run, since the real champion (`xgboost`) is tree-based but not in `NEEDS_DENSE`; caught
and fixed during code review before delivery. CatBoost's raw-categorical DataFrame path (if it were
ever champion after a future re-run) is passed to `TreeExplainer` as-is, per `shap`'s documented
CatBoost support. Computed on a bounded sample (150 test rows / 50 background rows) to stay
laptop-safe, stated explicitly rather than silently narrowed.

## Standing rules this notebook follows
- **Execution boundary / zero-fabrication**: Claude wrote this notebook; it does not run it. Every
  number is computed live during the real run — nothing here is copied from Gate 3's artifacts and
  relabeled.
- **HYPER**: Gate 3's feature-engineering code (Sections 5-7: Gold-layer reload, train/test split,
  shared one-hot/frequency preprocessing, CatBoost raw-categorical path) is reused verbatim so the
  paired CV re-run trains on the identical rows/columns Gate 3 used — required for the consistency
  check to be meaningful, not just convenient. `src/utils/bp1_config_sync.py` reused unmodified.
- **Continue gracefully on failure**: SHAP computation is wrapped in try/except — a failure is
  recorded plainly (`shap_error` in the output JSON) and the statistical-validation half still
  completes rather than halting entirely.
- **Idempotent**: re-running overwrites this gate's artifacts and its own `gate4_...` config block,
  without touching Gates 1-3's fields.

## Outputs (idempotent overwrite-in-place)
- `notebooks/bp2_customer_friction_classification/artifacts/gate4_statistical_validation.json`
- `notebooks/bp2_customer_friction_classification/artifacts/gate4_shap_top_features.csv`
- `notebooks/bp2_customer_friction_classification/artifacts/model_inventory_entry.json` (Gate 4
  fields added to the existing Gate 3 entry, in place)
- `configs/bp2_customer_friction_classification.yaml` — `gate4_statistical_validation` block
  appended/updated

## Prerequisites
BP2 Gate 3 must have been real-run at least once (this notebook reads its champion/runner-up from
`gate3_cv_benchmark_results.csv` and raises if that file is missing — confirmed present from the
real run already reported). The `shap` package must be installed — confirmed installed in this
project's environment during BP1 Gate 4 (not re-verified as still installed here beyond the live
`importlib.util.find_spec` check this notebook itself performs).

## If a structural check below fails
It raises `AssertionError` naming the failing check. If the CV-consistency check fails, this
notebook's feature-engineering code has drifted from Gate 3's — fix the drift, do not silence the
check.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP2 Gate 4 statistical validation / explainability notebook.
Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

import os, sys, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import (  # noqa: E402
    assert_within_ram_ceiling,
    configure_performance,
    load_resource_limits,
    memory_headroom_gb,
)

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import importlib.util  # noqa: E402
import re  # noqa: E402
import time  # noqa: E402
import yaml  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
from scipy import sparse as sp  # noqa: E402
from scipy import stats  # noqa: E402
from sklearn.compose import ColumnTransformer  # noqa: E402
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import f1_score, roc_auc_score  # noqa: E402
from sklearn.model_selection import StratifiedKFold, train_test_split  # noqa: E402
from sklearn.preprocessing import LabelEncoder, OneHotEncoder  # noqa: E402
from xgboost import XGBClassifier  # noqa: E402
from lightgbm import LGBMClassifier  # noqa: E402
from catboost import CatBoostClassifier  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

if importlib.util.find_spec("shap") is None:
    raise ImportError(
        "[CHECK FAILED] The 'shap' package is required for BP2 Gate 4 and was not confirmed installed. "
        "Run `pip install shap` (inside this project's own environment) before running this notebook - "
        "BP1 Gate 4 already hit and documented this exact prerequisite."
    )
import shap  # noqa: E402
print(f"[OK] shap {shap.__version__} confirmed installed (live check, not assumed).")

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp2_customer_friction_classification" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

GOLD_PATH = DATA_PROCESSED / "cfpb_friction_severity_gold.parquet"
BP2_CONFIG_PATH = CONFIGS_DIR / "bp2_customer_friction_classification.yaml"
SEVERITY_CONFIG_PATH = CONFIGS_DIR / "bp2_friction_severity_taxonomy.yaml"

for p in (GOLD_PATH, BP2_CONFIG_PATH, SEVERITY_CONFIG_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Required input not found: {p}. Confirm BP2 Gates 1-3 all completed for real."
        )

# ============================================================
# SECTION 4: Load Gate 3's real results - champion + runner-up read LIVE, never hardcoded here.
# Gate 3 only recorded mean/std per candidate - the paired test below needs individual fold scores,
# which is why this gate re-runs the identical CV split rather than reusing Gate 3's summary alone.
# ============================================================
with open(BP2_CONFIG_PATH, "r", encoding="utf-8") as f:
    bp2_config = yaml.safe_load(f)
with open(SEVERITY_CONFIG_PATH, "r", encoding="utf-8") as f:
    severity_config = yaml.safe_load(f)

TARGET_COL = bp2_config["target_definition"]["primary_target"]
RANDOM_STATE = bp2_config["random_state"]
ORDINAL_CLASSES = list(severity_config["severity_classes"].keys())
N_CLASSES = len(ORDINAL_CLASSES)

gate3_block = bp2_config.get("gate3_model_benchmark")
assert gate3_block is not None, (
    "[CHECK FAILED] gate3_model_benchmark is missing from configs/bp2_customer_friction_classification.yaml "
    "- run BP2 Gate 3 first."
)

gate3_cv_csv_path = ARTIFACTS_DIR / "gate3_cv_benchmark_results.csv"
assert gate3_cv_csv_path.exists(), (
    f"[CHECK FAILED] {gate3_cv_csv_path} not found - run BP2 Gate 3 first (it writes this file)."
)
gate3_cv_df = pd.read_csv(gate3_cv_csv_path)
# kind="mergesort" (stable) so an exact tie in mean_f1_macro breaks deterministically by the candidates'
# original CSV row order, rather than an unstable-sort's unpredictable tie order (BP1 Lesson #16).
passing = gate3_cv_df[gate3_cv_df["status"] == "OK"].sort_values("mean_f1_macro", ascending=False, kind="mergesort")
assert len(passing) >= 2, (
    "[CHECK FAILED] Fewer than 2 candidates passed in Gate 3 - cannot run a paired champion-vs-runner-up "
    "comparison. Re-check Gate 3's real run."
)
CHAMPION_NAME = passing.iloc[0]["model"]
RUNNER_UP_NAME = passing.iloc[1]["model"]
gate3_champion_recorded_f1 = float(passing.iloc[0]["mean_f1_macro"])
assert CHAMPION_NAME == gate3_block["champion_model"], (
    f"[CHECK FAILED] Champion mismatch: gate3_cv_benchmark_results.csv says '{CHAMPION_NAME}' but "
    f"configs/bp2_customer_friction_classification.yaml's gate3_model_benchmark.champion_model says "
    f"'{gate3_block['champion_model']}' - these must agree; re-run Gate 3."
)
print(f"[OK] Gate 3 champion (live, re-verified): {CHAMPION_NAME} (recorded mean CV f1_macro={gate3_champion_recorded_f1})")
print(f"[OK] Gate 3 runner-up (live, for paired comparison): {RUNNER_UP_NAME} "
      f"(recorded mean CV f1_macro={float(passing.iloc[1]['mean_f1_macro'])})")
if len(gate3_cv_df) > len(passing):
    failed_names = gate3_cv_df.loc[gate3_cv_df["status"] != "OK", "model"].tolist()
    print(f"[NOTE] Gate 3 candidate(s) that failed and are excluded from champion/runner-up eligibility: "
          f"{failed_names} (open item - failure cause not investigated at the user's explicit instruction; "
          "does not affect this gate's validity since only passing candidates are ever eligible).")

hw_summary_path = CONFIGS_DIR / "hardware_benchmark_summary.json"
with open(hw_summary_path, "r", encoding="utf-8") as f:
    hw_summary = json.load(f)
cv_settings = RESOURCE_LIMITS["cv"]
RNG = np.random.RandomState(cv_settings["random_state"])

# ============================================================
# SECTION 5: Rebuild the real Gold-layer feature frame EXACTLY as Gate 3 did (HYPER: identical
# construction reused, not redesigned) - this gate's paired CV re-run is only valid if it trains on
# the IDENTICAL rows/columns Gate 3 used, or the champion-mean consistency check below would fail
# for a reason that has nothing to do with statistical validity.
# ============================================================
FEATURE_COLS_CATEGORICAL = ["Product", "Sub-product", "Issue", "Sub-issue", "State",
                            "Submitted via", "common_taxonomy_bucket"]
COMPANY_COL = "Company"
BARRED_COLUMNS = ["Company response to consumer", "Timely response?", "Date received",
                  "Date sent to company", "Company public response", "Complaint ID", "ZIP code", "Tags"]

gold_lazy = pl.scan_parquet(GOLD_PATH)
select_cols = FEATURE_COLS_CATEGORICAL + [COMPANY_COL, TARGET_COL]
df_pl = (
    gold_lazy.select(select_cols)
    .filter(pl.col(TARGET_COL).is_in(ORDINAL_CLASSES))
    .collect()
)
for barred in BARRED_COLUMNS:
    assert barred not in df_pl.columns, f"[CHECK FAILED] barred column '{barred}' present in the loaded feature frame."
print(f"[OK] Reloaded real Gold layer, trainable rows: {df_pl.height:,} (must match Gate 3's row count).")

feature_data = {col: df_pl[col].cast(pl.Utf8).fill_null("MISSING").to_list() for col in FEATURE_COLS_CATEGORICAL}
feature_data[COMPANY_COL] = df_pl[COMPANY_COL].cast(pl.Utf8).fill_null("MISSING").to_list()
target_data = df_pl[TARGET_COL].cast(pl.Utf8).to_list()
X_full = pd.DataFrame(feature_data)
y_full_labels = pd.Series(target_data, name=TARGET_COL)

# ============================================================
# SECTION 6: Identical stratified train/test split as Gate 3 (same test_size/stratify/random_state)
# ============================================================
X_train_raw, X_test_raw, y_train_labels, y_test_labels = train_test_split(
    X_full, y_full_labels, test_size=0.20, stratify=y_full_labels, random_state=RANDOM_STATE
)
label_encoder = LabelEncoder().fit(y_train_labels)
y_train = label_encoder.transform(y_train_labels)
y_test = label_encoder.transform(y_test_labels)
print(f"[OK] Reproduced Gate 3's train/test split: train={len(X_train_raw):,}, test={len(X_test_raw):,}.")

# ============================================================
# SECTION 7: Rebuild the shared preprocessing EXACTLY as Gate 3 (fit on TRAIN only) - one-hot for
# 5 of 6 candidates + frequency-encoded Company; CatBoost gets the raw categorical columns instead.
# ============================================================
ohe = ColumnTransformer(
    [("ohe", OneHotEncoder(handle_unknown="ignore", dtype=np.float32), FEATURE_COLS_CATEGORICAL)],
    remainder="drop",
)
X_train_ohe = ohe.fit_transform(X_train_raw)
X_test_ohe = ohe.transform(X_test_raw)

company_freq_map = X_train_raw[COMPANY_COL].value_counts().to_dict()
train_company_freq = X_train_raw[COMPANY_COL].map(company_freq_map).fillna(0).to_numpy(dtype=np.float32).reshape(-1, 1)
test_company_freq = X_test_raw[COMPANY_COL].map(company_freq_map).fillna(0).to_numpy(dtype=np.float32).reshape(-1, 1)

X_train_shared = sp.hstack([X_train_ohe, sp.csr_matrix(train_company_freq)], format="csr")
X_test_shared = sp.hstack([X_test_ohe, sp.csr_matrix(test_company_freq)], format="csr")

CATBOOST_FEATURE_COLS = FEATURE_COLS_CATEGORICAL + [COMPANY_COL]
X_train_cat = X_train_raw[CATBOOST_FEATURE_COLS].copy()
X_test_cat = X_test_raw[CATBOOST_FEATURE_COLS].copy()
CATBOOST_CAT_FEATURE_INDICES = list(range(len(CATBOOST_FEATURE_COLS)))
print(f"[OK] Reproduced Gate 3's shared feature matrix: train={X_train_shared.shape}, test={X_test_shared.shape}.")

# ============================================================
# SECTION 8: Candidate definitions - MUST exactly mirror Gate 3's (single-source-of-truth risk,
# guarded by the consistency check in Section 10 below - if these two notebooks' definitions ever
# drift apart, that check catches it rather than silently producing an incomparable "champion").
# ============================================================
CANDIDATES = {
    "logistic_regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=cv_settings["random_state"]
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=100, max_depth=20, class_weight="balanced", n_jobs=1,
        random_state=cv_settings["random_state"],
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=100, random_state=cv_settings["random_state"]
    ),
    "xgboost": XGBClassifier(
        n_estimators=100, max_depth=6, n_jobs=1, verbosity=0, random_state=cv_settings["random_state"]
    ),
    "lightgbm": LGBMClassifier(
        n_estimators=100, class_weight="balanced", n_jobs=1, verbose=-1,
        random_state=cv_settings["random_state"],
    ),
    "catboost": CatBoostClassifier(
        iterations=100, thread_count=1, verbose=False, allow_writing_files=False,
        auto_class_weights="Balanced", random_state=cv_settings["random_state"],
        cat_features=CATBOOST_CAT_FEATURE_INDICES,
    ),
}
NEEDS_DENSE = {"hist_gradient_boosting"}
USES_RAW_CATEGORICAL = {"catboost"}

def _cv_inputs(name):
    if name in USES_RAW_CATEGORICAL:
        return X_train_cat, y_train_labels.to_numpy()
    X_cv = X_train_shared
    if name in NEEDS_DENSE:
        X_cv = np.asarray(X_cv.todense(), dtype=np.float32)
    return X_cv, y_train

# ============================================================
# SECTION 9: Re-run the IDENTICAL CV split for champion + runner-up ONLY, capturing PER-FOLD scores.
# Sequential, one model/one fold at a time - unlike Gate 3's benchmark loop this has NO joblib
# concurrency at all (models are fit one at a time here), so the concurrency-driven memory risk
# Lesson #21 addressed does not apply to this loop by construction. Still applies the same WARP
# monitoring discipline (live headroom logged, ceiling re-checked every fold, densified per-fold
# matrices explicitly freed) since this dataset is the same real scale that crashed before, and a
# brief pause is inserted between the two candidates as the same precautionary thermal pacing
# Gate 3 now uses, given each candidate here is a FULL real-scale 5-fold fit, not a fast synthetic one.
# ============================================================
skf = StratifiedKFold(
    n_splits=cv_settings["n_splits"], shuffle=cv_settings["shuffle"], random_state=cv_settings["random_state"]
)
fold_scores = {}
for cand_idx, name in enumerate((CHAMPION_NAME, RUNNER_UP_NAME)):
    model_template = CANDIDATES[name]
    X_cv_full, y_cv_full = _cv_inputs(name)
    print(f"\n[GATE4] Re-running identical {cv_settings['n_splits']}-fold CV for {name} "
          f"({cand_idx + 1}/2) to capture per-fold scores (sequential, no concurrency)...")
    t0 = time.perf_counter()
    scores = []
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(X_train_shared, y_train), start=1):
        headroom = memory_headroom_gb(RESOURCE_LIMITS["ceilings"]["max_ram_fraction"])
        if name in USES_RAW_CATEGORICAL:
            X_fold_train, X_fold_val = X_cv_full.iloc[train_idx], X_cv_full.iloc[val_idx]
        else:
            X_fold_train, X_fold_val = X_cv_full[train_idx], X_cv_full[val_idx]
        y_fold_train, y_fold_val = y_cv_full[train_idx], y_cv_full[val_idx]
        fold_model = type(model_template)(**model_template.get_params())
        fold_model.fit(X_fold_train, y_fold_train)
        fold_pred = fold_model.predict(X_fold_val)
        fold_f1 = f1_score(y_fold_val, fold_pred, average="macro", zero_division=0)
        scores.append(fold_f1)
        print(f"  fold {fold_i}/{cv_settings['n_splits']}: f1_macro={fold_f1:.4f} (headroom before fold: {headroom}GB)")
        del fold_model, X_fold_train, X_fold_val
        assert_within_ram_ceiling(RESOURCE_LIMITS)
    del X_cv_full
    fold_scores[name] = np.array(scores)
    print(f"[GATE4] {name} done in {time.perf_counter() - t0:.1f}s - mean={np.mean(scores):.4f}, std={np.std(scores):.4f}")
    if cand_idx == 0:
        print("[WARP] cooling down 15s before the runner-up's full re-fit (precautionary thermal pacing, "
              "same rationale as Gate 3's Lesson #21 hardening - not a measured reading)...")
        time.sleep(15)

# ============================================================
# SECTION 10: Consistency check - this notebook's own recomputed champion mean must match Gate 3's
# recorded value within tolerance, or the two notebooks' definitions have drifted apart.
# ============================================================
recomputed_champion_mean = float(np.mean(fold_scores[CHAMPION_NAME]))
consistency_diff = abs(recomputed_champion_mean - gate3_champion_recorded_f1)
print(f"\n[CHECK] Recomputed champion mean CV f1_macro: {recomputed_champion_mean:.4f} "
      f"(Gate 3 recorded: {gate3_champion_recorded_f1:.4f}, diff={consistency_diff:.4f})")

# ============================================================
# SECTION 11: Paired significance test - champion vs runner-up, on the paired fold scores
# ============================================================
champion_scores = fold_scores[CHAMPION_NAME]
runnerup_scores = fold_scores[RUNNER_UP_NAME]
paired_diffs = champion_scores - runnerup_scores

if np.allclose(paired_diffs, 0.0):
    ttest_stat, ttest_p = None, None
    print(f"\n[LIMITATION] {CHAMPION_NAME} and {RUNNER_UP_NAME} scored identically on every fold - a paired "
          "t-test is undefined (zero variance in the differences).")
else:
    ttest_result = stats.ttest_rel(champion_scores, runnerup_scores)
    ttest_stat, ttest_p = float(ttest_result.statistic), float(ttest_result.pvalue)
    print(f"\n[RESULT] Paired t-test ({CHAMPION_NAME} vs {RUNNER_UP_NAME}, n={cv_settings['n_splits']} folds): "
          f"t={ttest_stat:.3f}, p={ttest_p:.4f}")
    print(f"[LIMITATION] n={cv_settings['n_splits']} folds is a very small sample for a t-test - treat this "
          "p-value as directional evidence, not a high-powered statistical claim.")

try:
    if np.allclose(paired_diffs, 0.0):
        raise ValueError("all paired differences are zero")
    wilcoxon_result = stats.wilcoxon(champion_scores, runnerup_scores)
    wilcoxon_stat, wilcoxon_p = float(wilcoxon_result.statistic), float(wilcoxon_result.pvalue)
except ValueError as e:
    wilcoxon_stat, wilcoxon_p = None, None
    print(f"[LIMITATION] Wilcoxon signed-rank test could not run: {e}")

# ============================================================
# SECTION 12: Refit champion on FULL train, get held-out test predictions - bootstrap CI + ROC-AUC
# ============================================================
assert_within_ram_ceiling(RESOURCE_LIMITS)
if CHAMPION_NAME in USES_RAW_CATEGORICAL:
    X_train_final, X_test_final = X_train_cat, X_test_cat
    y_train_final = y_train_labels.to_numpy()
else:
    X_train_final, X_test_final = X_train_shared, X_test_shared
    if CHAMPION_NAME in NEEDS_DENSE:
        X_train_final = np.asarray(X_train_final.todense(), dtype=np.float32)
        X_test_final = np.asarray(X_test_final.todense(), dtype=np.float32)
    y_train_final = y_train

champion_model = CANDIDATES[CHAMPION_NAME]
print(f"\n[GATE4] Refitting champion ({CHAMPION_NAME}) on the full train split for bootstrap CI + ROC-AUC...")
champion_model.fit(X_train_final, y_train_final)

if CHAMPION_NAME in USES_RAW_CATEGORICAL:
    y_pred_raw = champion_model.predict(X_test_final)
    y_pred_encoded = label_encoder.transform(np.asarray(y_pred_raw).ravel())
else:
    y_pred_encoded = np.asarray(champion_model.predict(X_test_final)).astype(int).ravel()

point_test_f1_macro = f1_score(y_test, y_pred_encoded, average="macro", zero_division=0)

N_BOOTSTRAP = 1000
n_test = len(y_test)
y_test_arr = np.asarray(y_test)
y_pred_arr = np.asarray(y_pred_encoded)
boot_scores = np.empty(N_BOOTSTRAP)
for b in range(N_BOOTSTRAP):
    idx = RNG.randint(0, n_test, size=n_test)
    boot_scores[b] = f1_score(y_test_arr[idx], y_pred_arr[idx], average="macro", zero_division=0)
ci_low, ci_high = float(np.percentile(boot_scores, 2.5)), float(np.percentile(boot_scores, 97.5))
print(f"[RESULT] Held-out test f1_macro: {point_test_f1_macro:.4f}, "
      f"95% bootstrap CI ({N_BOOTSTRAP} resamples): [{ci_low:.4f}, {ci_high:.4f}]")

roc_auc_ovr_macro = None
if hasattr(champion_model, "predict_proba"):
    if CHAMPION_NAME in USES_RAW_CATEGORICAL:
        y_proba_raw = champion_model.predict_proba(X_test_final)
        proba_classes = list(champion_model.classes_)
        proba_order = [proba_classes.index(c) for c in label_encoder.classes_]
        y_proba = np.asarray(y_proba_raw)[:, proba_order]
    else:
        y_proba = champion_model.predict_proba(X_test_final)
        pipeline_classes = list(champion_model.classes_)
        assert pipeline_classes == list(range(N_CLASSES)), (
            f"[CHECK FAILED] Champion's class order {pipeline_classes} does not match the expected "
            f"0..{N_CLASSES - 1} integer-encoded order - ROC-AUC column alignment would be wrong."
        )
    roc_auc_ovr_macro = float(roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro"))
    print(f"[RESULT] {N_CLASSES}-class one-vs-rest macro ROC-AUC: {roc_auc_ovr_macro:.4f}")
else:
    print(f"[LIMITATION] {CHAMPION_NAME} has no predict_proba - ROC-AUC not computable for this champion.")

# ============================================================
# SECTION 13: SHAP explainability - explainer chosen by the champion's actual model type, on a
# bounded sample (never the full held-out set, to stay laptop-safe under WARP's ceilings).
# ============================================================
SHAP_SAMPLE_SIZE = min(150, len(X_test_raw))
SHAP_BACKGROUND_SIZE = min(50, len(X_train_raw))
shap_top_features = None
shap_error = None
try:
    sample_idx = RNG.choice(len(X_test_raw), size=SHAP_SAMPLE_SIZE, replace=False)
    bg_idx = RNG.choice(len(X_train_raw), size=SHAP_BACKGROUND_SIZE, replace=False)
    is_linear_champion = isinstance(champion_model, LogisticRegression)

    if CHAMPION_NAME in USES_RAW_CATEGORICAL:
        X_sample = X_test_cat.iloc[sample_idx]
        X_bg = X_train_cat.iloc[bg_idx]
        feature_names = np.array(CATBOOST_FEATURE_COLS)
    else:
        X_sample_vec = X_test_shared[sample_idx]
        X_bg_vec = X_train_shared[bg_idx]
        feature_names = np.array(list(ohe.get_feature_names_out()) + ["Company_freq"])
        # Real environment finding already established in BP1 Gate 4: shap.TreeExplainer.shap_values()
        # rejects a sparse matrix outright for EVERY tree-based candidate, not only the ones Gate 3's
        # own NEEDS_DENSE set names (that set is about a FIT-time constraint - a different scope). So
        # the sample/background fed to TreeExplainer is always densified here; only the
        # LogisticRegression/LinearExplainer path is left sparse (confirmed to accept sparse input
        # directly, same as BP1 Gate 4's own finding).
        if not is_linear_champion:
            X_sample_vec = np.asarray(X_sample_vec.todense(), dtype=np.float32)
            X_bg_vec = np.asarray(X_bg_vec.todense(), dtype=np.float32)
        X_sample, X_bg = X_sample_vec, X_bg_vec

    if is_linear_champion:
        print(f"\n[GATE4] SHAP: using LinearExplainer for {CHAMPION_NAME} "
              f"(sample={SHAP_SAMPLE_SIZE} rows, background={SHAP_BACKGROUND_SIZE} rows)...")
        explainer = shap.LinearExplainer(champion_model, X_bg)
        shap_values = explainer.shap_values(X_sample)
    else:
        # Real environment quirk already found in BP1 Gate 4: shap.TreeExplainer.shap_values()
        # rejects a sparse matrix outright for every tree-based candidate except CatBoost's own raw
        # categorical path - densified above for the shared-feature branch. CatBoost's raw
        # categorical DataFrame is passed through as-is (shap's documented CatBoost support).
        print(f"\n[GATE4] SHAP: using TreeExplainer for {CHAMPION_NAME} "
              f"(sample={SHAP_SAMPLE_SIZE} rows, background={SHAP_BACKGROUND_SIZE} rows)...")
        explainer = shap.TreeExplainer(champion_model)
        shap_values = explainer.shap_values(X_sample)

    if isinstance(shap_values, list):
        abs_vals = np.mean([np.abs(np.asarray(sv)) for sv in shap_values], axis=0)
    else:
        arr = np.asarray(shap_values)
        abs_vals = np.abs(arr).mean(axis=-1) if arr.ndim == 3 else np.abs(arr)

    mean_abs_shap = np.asarray(abs_vals).mean(axis=0).ravel()
    assert len(mean_abs_shap) == len(feature_names), (
        f"[CHECK FAILED] SHAP feature-importance length ({len(mean_abs_shap)}) does not match "
        f"the feature name count ({len(feature_names)})."
    )
    top_idx = np.argsort(mean_abs_shap)[::-1][:20]
    shap_top_features = pd.DataFrame({
        "feature": feature_names[top_idx],
        "mean_abs_shap": mean_abs_shap[top_idx],
    })
    print(f"\n[RESULT] Top 10 globally important features (mean |SHAP value|, sampled, champion={CHAMPION_NAME}):")
    print(shap_top_features.head(10).to_string(index=False))
except Exception as e:  # noqa: BLE001 - continue gracefully; statistical validation above still completes
    shap_error = f"{type(e).__name__}: {e}"
    print(f"[LIMITATION] SHAP computation failed for champion model family "
          f"'{type(CANDIDATES[CHAMPION_NAME]).__name__}': {shap_error}. Statistical validation results "
          "above are unaffected and still valid.")

# ============================================================
# SECTION 14: Write outputs (idempotent overwrite-in-place)
# ============================================================
stat_validation = {
    "bp_id": "bp2",
    "gate": 4,
    "champion_model": CHAMPION_NAME,
    "runner_up_model": RUNNER_UP_NAME,
    "champion_fold_f1_macro": [round(float(s), 4) for s in champion_scores],
    "runner_up_fold_f1_macro": [round(float(s), 4) for s in runnerup_scores],
    "recomputed_champion_mean_cv_f1_macro": round(recomputed_champion_mean, 4),
    "gate3_recorded_champion_mean_cv_f1_macro": round(gate3_champion_recorded_f1, 4),
    "consistency_check_diff": round(consistency_diff, 6),
    "paired_ttest_statistic": round(ttest_stat, 4) if ttest_stat is not None else None,
    "paired_ttest_pvalue": round(ttest_p, 4) if ttest_p is not None else None,
    "wilcoxon_statistic": round(wilcoxon_stat, 4) if wilcoxon_stat is not None else None,
    "wilcoxon_pvalue": round(wilcoxon_p, 4) if wilcoxon_p is not None else None,
    "statistical_test_limitation": f"n={cv_settings['n_splits']} CV folds is a small sample - treat "
                                    "p-values as directional evidence, not a high-powered statistical claim.",
    "held_out_test_f1_macro_point_estimate": round(float(point_test_f1_macro), 4),
    "held_out_test_f1_macro_bootstrap_ci_95": [round(ci_low, 4), round(ci_high, 4)],
    "bootstrap_n_iterations": N_BOOTSTRAP,
    "roc_auc_ovr_macro": round(roc_auc_ovr_macro, 4) if roc_auc_ovr_macro is not None else None,
    "n_classes": N_CLASSES,
    "gate3_failed_candidates_excluded": gate3_cv_df.loc[gate3_cv_df["status"] != "OK", "model"].tolist(),
    "shap_sample_size": SHAP_SAMPLE_SIZE,
    "shap_background_size": SHAP_BACKGROUND_SIZE,
    "shap_error": shap_error,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
stat_path = ARTIFACTS_DIR / "gate4_statistical_validation.json"
with open(stat_path, "w", encoding="utf-8") as f:
    json.dump(stat_validation, f, indent=2)
print(f"\n[SAVED] {stat_path.relative_to(PROJECT_ROOT)}")

shap_csv_path = ARTIFACTS_DIR / "gate4_shap_top_features.csv"
if shap_top_features is not None:
    shap_top_features.to_csv(shap_csv_path, index=False)
    print(f"[SAVED] {shap_csv_path.relative_to(PROJECT_ROOT)}")
else:
    pd.DataFrame({"feature": [], "mean_abs_shap": []}).to_csv(shap_csv_path, index=False)
    print(f"[SAVED] {shap_csv_path.relative_to(PROJECT_ROOT)} (empty - SHAP failed, see shap_error in the JSON)")

inventory_path = ARTIFACTS_DIR / "model_inventory_entry.json"
if inventory_path.exists():
    with open(inventory_path, "r", encoding="utf-8") as f:
        model_inventory_entry = json.load(f)
else:
    model_inventory_entry = {"bp_id": "bp2", "model_name": CHAMPION_NAME}
model_inventory_entry["status"] = "Gate 4 statistical validation + explainability complete"
model_inventory_entry["gate4_paired_ttest_pvalue_vs_runner_up"] = round(ttest_p, 4) if ttest_p is not None else None
model_inventory_entry["gate4_held_out_test_f1_macro_bootstrap_ci_95"] = [round(ci_low, 4), round(ci_high, 4)]
model_inventory_entry["gate4_roc_auc_ovr_macro"] = round(roc_auc_ovr_macro, 4) if roc_auc_ovr_macro is not None else None
model_inventory_entry["gate4_generated_at_utc"] = datetime.now(timezone.utc).isoformat()
with open(inventory_path, "w", encoding="utf-8") as f:
    json.dump(model_inventory_entry, f, indent=2)
print(f"[SAVED] {inventory_path.relative_to(PROJECT_ROOT)} (Gate 4 fields added)")

from utils.bp1_config_sync import write_gate_block  # noqa: E402

status_text = BP2_CONFIG_PATH.read_text(encoding="utf-8")
current_status_line = [ln for ln in status_text.splitlines() if ln.startswith("status:")][0]
new_status_value = current_status_line.split('"')[1] + "_gate4_confirmed" \
    if "_gate4_confirmed" not in current_status_line else current_status_line.split('"')[1]
status_text = re.sub(r'^status:.*$', f'status: "{new_status_value}"', status_text, count=1, flags=re.MULTILINE)
BP2_CONFIG_PATH.write_text(status_text, encoding="utf-8")

gate4_marker = "# --- Gate 4 (Statistical Validation & Explainability) results (appended, idempotent overwrite) ---"
gate4_block_lines = [
    "gate4_statistical_validation:",
    f'  champion_model: "{CHAMPION_NAME}"',
    f'  runner_up_model: "{RUNNER_UP_NAME}"',
    f"  paired_ttest_pvalue: {round(ttest_p, 4) if ttest_p is not None else 'null'}",
    f"  held_out_test_f1_macro_bootstrap_ci_95: [{round(ci_low, 4)}, {round(ci_high, 4)}]",
    f"  roc_auc_ovr_macro: {round(roc_auc_ovr_macro, 4) if roc_auc_ovr_macro is not None else 'null'}",
    f'  generated_at_utc: "{datetime.now(timezone.utc).isoformat()}"',
]
write_gate_block(BP2_CONFIG_PATH, gate4_marker, gate4_block_lines)
print(f"[SAVED] {BP2_CONFIG_PATH.relative_to(PROJECT_ROOT)} (gate4_statistical_validation block)")

# ============================================================
# SECTION 15: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "champion_runner_up_read_live_from_gate3_results": CHAMPION_NAME != RUNNER_UP_NAME,
    "identical_cv_split_used_as_gate3": True,  # by construction - same StratifiedKFold params, Section 9
    "cv_consistency_check_within_tolerance": consistency_diff < 0.01,
    "paired_significance_test_computed": len(champion_scores) == cv_settings["n_splits"] and len(runnerup_scores) == cv_settings["n_splits"],
    "bootstrap_ci_computed": True,  # computed regardless of point-estimate containment, Section 12
    "roc_auc_computed_or_explicitly_unavailable": True,  # either a float or None-with-printed-reason
    "no_barred_column_in_feature_frame": all(b not in df_pl.columns for b in BARRED_COLUMNS),
    "statistical_validation_json_written": stat_path.exists(),
    "shap_csv_written": shap_csv_path.exists(),
    "model_inventory_updated": inventory_path.exists(),
    "bp2_config_yaml_updated": BP2_CONFIG_PATH.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(f"\n[ALL CHECKS PASSED] BP2 Gate 4 complete. Champion {CHAMPION_NAME} vs runner-up {RUNNER_UP_NAME}: "
      f"paired t-test p={round(ttest_p, 4) if ttest_p is not None else 'undefined (identical fold scores)'} "
      f"(n={cv_settings['n_splits']} folds - directional, not definitive). "
      f"Held-out test f1_macro={round(float(point_test_f1_macro), 4)}, "
      f"95% bootstrap CI=[{round(ci_low, 4)}, {round(ci_high, 4)}]. "
      f"{'SHAP: OK' if shap_error is None else f'SHAP: FAILED ({shap_error})'}. "
      "Proceed to BP2 Gate 5 (Decision Layer & Reporting) next.")
